In [0]:
from pyspark.sql.functions import *
storage_path = "abfss://capstonet1container@capstonet1.dfs.core.windows.net/landing/adls"
schema_base = "/Volumes/capstone_team1/bronze/raw_data/schemas"
checkpoint_base = "/Volumes/capstone_team1/bronze/raw_data/checkpoints"

datasets = {
    "primary_events": "bronze_primary_events",
    "secondary_events": "bronze_secondary_events",
    "status_updates": "bronze_status_updates"
}

# Auto Loader ingestion
for folder_name, table_name in datasets.items():
    source_path = f"{storage_path}/{folder_name}"
    df = (spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "csv")
          .option("header", "true")
          .option("cloudFiles.schemaLocation",f"{schema_base}/{folder_name}")
          .load(source_path)
    )

    # Add ingestion metadata
    df = (df
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_ingestion_timestamp", current_timestamp())
    )

    # Append into Bronze Delta table
    query = (df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation",
                f"{checkpoint_base}/{folder_name}")
        .trigger(availableNow=True)
        .toTable(table_name)
    )

    query.awaitTermination()

    print(f"Successfully ingested {folder_name} into {table_name}")